# Stage 9 — Argued figures (post Stage 1–8 EDA)

Six annotated figures from cached impact-EDA outputs. Regenerate Stage 1–8 artefacts first if needed:

```powershell
python -m archive.impact_eda.impact_stages --through 8
```

PNG outputs: `eda/outputs/impact_eda/figures/stage9/`.


In [ ]:
# Setup
from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd

from processing_data.paths import COURSE_RAW, EXT_DATA, PROCESSED_DIR, ROOT

if not (Path.cwd() / "archive" / "impact_eda" / "impact_stages.py").exists():
    import os
    os.chdir(ROOT)

OUT = ROOT / "eda" / "outputs" / "impact_eda"
FIG_DIR = OUT / "figures" / "stage9"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ADMIN2_GEO = COURSE_RAW / "Administrative boundaries" / "ssd_admin2.geojson"
assert OUT.exists(), "Run: python -m archive.impact_eda.impact_stages --through 8"


def sp(a: pd.Series, b: pd.Series) -> float:
    """Spearman via ranks (no scipy)."""
    return a.rank().corr(b.rank())


def save_fig(fig, name: str, claim: str, label: str = "Claim") -> Path:
    fig.suptitle(f"{label}: {claim}", fontsize=10, y=1.02)
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", path.relative_to(ROOT))
    return path


def load_fews_ha_panel() -> pd.DataFrame:
    xw = pd.read_csv(OUT / "crosswalks" / "fews_fnid_to_admin2.csv")
    crop = pd.read_csv(EXT_DATA / "FEWS_crop_data" / "crop_data.csv")
    ah = crop[crop.indicator == "Area Harvested"].copy()
    ah["year"] = ah.season_year.str.extract(r"(\d{4})").astype(int)
    ah = ah.merge(xw[["fnid", "cod_pcode"]].drop_duplicates(), on="fnid", how="left")
    ah = ah.dropna(subset=["cod_pcode"])
    return ah.groupby(["cod_pcode", "year"], as_index=False)["value"].sum().rename(
        columns={"cod_pcode": "adm2_pcode", "value": "ha"}
    )


def flood_fews_merged() -> tuple[pd.DataFrame, pd.DataFrame]:
    cov = pd.read_csv(OUT / "flood_tile_coverage_admin2.csv")
    un = pd.read_csv(OUT / "admin2_unusual_flood_2022_2024.csv")
    rec = pd.read_csv(OUT / "admin2_recurring_flood_2022_2024.csv")
    ha = load_fews_ha_panel()
    m = un.merge(ha, on=["adm2_pcode", "year"], how="inner").merge(
        cov[["adm2_pcode", "flood_tile_coverage_share"]], on="adm2_pcode", how="left"
    )
    mr = rec.merge(ha, on=["adm2_pcode", "year"], how="inner").merge(
        cov[["adm2_pcode", "flood_tile_coverage_share"]], on="adm2_pcode", how="left"
    )
    return m, mr


plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "#f8f8f8"})


## Figure 1 — One Stage 7 statistic is an artefact; the correlation is separate

**Claim:** The ASAP crop contrast (9.7% vs 0.5% by unusual-flood presence) is driven by exactly two unobservable counties (Manyo, Renk), not by cropland avoiding floods.

**Falsifier:** If many counties with zero unusual pixels had middling crop cover, the group mean would still be informative — they do not.


In [ ]:
# Figure 1
cov = pd.read_csv(OUT / "flood_tile_coverage_admin2.csv")
asap = pd.read_csv(OUT / "stage6_asap_zonal_admin2.csv")
unusual = pd.read_csv(OUT / "admin2_unusual_flood_2022_2024.csv")
has_unusual = unusual.groupby("adm2_pcode").unusual_unique_px.sum().gt(0)
asap = asap.assign(has_unusual_px=asap.adm2_pcode.map(has_unusual).fillna(False))
asap["crop_rank"] = asap.crop_mean_pct.rank(ascending=False, method="min").astype(int)

gdf = gpd.read_file(ADMIN2_GEO).to_crs("EPSG:4326")
gdf = gdf.merge(cov[["adm2_pcode", "flood_tile_coverage_share"]], on="adm2_pcode", how="left")
gdf = gdf.merge(asap[["adm2_pcode", "crop_mean_pct", "crop_rank"]], on="adm2_pcode", how="left")

fig, axes = plt.subplots(1, 2, figsize=(13, 7), gridspec_kw={"width_ratios": [1.15, 1]})

ax = axes[0]
gdf.plot(
    column="flood_tile_coverage_share",
    ax=ax,
    legend=True,
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    legend_kwds={"label": "MODIS flood tile coverage share", "shrink": 0.7},
)
zero = gdf[gdf.flood_tile_coverage_share.fillna(0) <= 0]
if len(zero):
    zero.plot(ax=ax, facecolor="none", edgecolor="#c0392b", hatch="///", linewidth=1.2)
for _, row in zero.iterrows():
    ax.annotate(
        f"{row.adm2_name}\n#{int(row.crop_rank)} crop, 0% tiles",
        xy=(row.geometry.centroid.x, row.geometry.centroid.y),
        fontsize=8,
        ha="center",
        color="#922b21",
        fontweight="bold",
    )
ax.set_title("Flood mask observability")
ax.set_axis_off()

ax = axes[1]
colors = {True: "#2980b9", False: "#c0392b"}
rng = np.random.default_rng(42)
for flag, sub in asap.groupby("has_unusual_px", observed=True):
    jitter = rng.uniform(-0.08, 0.08, len(sub))
    ax.scatter(
        np.full(len(sub), 1 if flag else 0) + jitter,
        sub.crop_mean_pct,
        alpha=0.65,
        s=28,
        c=colors[bool(flag)],
        label=f"any unusual px 2022–24 (n={len(sub)})" if flag else f"zero unusual px (n={len(sub)})",
    )
means = asap.groupby("has_unusual_px")["crop_mean_pct"].mean()
for i, (flag, val) in enumerate(means.items()):
    ax.axhline(val, xmin=0.02 + i * 0.48, xmax=0.46 + i * 0.48, color=colors[bool(flag)], lw=2)
    ax.text(0.52 + i * 0.48, val, f"Stage 7 mean={val:.2f}%", va="center", fontsize=9)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No unusual\nflood pixels", "Has unusual\nflood pixels"])
ax.set_ylabel("ASAP zonal mean crop % (all admin2)")
ax.set_title("ASAP crop by unusual-flood presence")
ax.legend(loc="upper right", fontsize=8)
ax.set_xlim(-0.35, 1.35)

save_fig(
    fig,
    "fig01_coverage_artefact_vs_asap_crop.png",
    "ASAP group contrast is a 2-county observability artefact (Manyo #3, Renk #1 crop); not evidence crops avoid floods.",
    label="Artefact",
)


## Figure 2 — Flood–harvest correlation survives coverage filtering

**Claim:** Dropping poorly tiled counties barely moves Spearman r between unusual flood pixels and FEWS harvested area; the unusual vs recurring sign flip is stable.

**Falsifier:** If r collapsed toward zero when coverage ≥ 0.95, observability would explain the negative association — it does not.


In [ ]:
# Figure 2
m, mr = flood_fews_merged()
years = [2022, 2023, 2024]
rows = []
for y in years:
    a = m[m.year == y]
    b = a[a.flood_tile_coverage_share >= 0.95]
    rows.append(("unusual", y, len(a), sp(a.unusual_unique_px, a.ha), len(b), sp(b.unusual_unique_px, b.ha)))
    a = mr[mr.year == y]
    b = a[a.flood_tile_coverage_share >= 0.95]
    rows.append(("recurring", y, len(a), sp(a.recurring_unique_px, a.ha), len(b), sp(b.recurring_unique_px, b.ha)))
df = pd.DataFrame(rows, columns=["mask", "year", "n_all", "r_all", "n_cov", "r_cov"])
print(df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, mask_kind in zip(axes, ["unusual", "recurring"]):
    sub = df[df["mask"] == mask_kind]
    x = np.arange(len(years))
    for i, y in enumerate(years):
        row = sub[sub.year == y].iloc[0]
        ax.plot([i - 0.12, i + 0.12], [row.r_all, row.r_cov], "o-", color="#2c3e50", lw=2, markersize=8)
        ax.text(i - 0.12, row.r_all, f"{row.r_all:.3f}", ha="right", va="bottom", fontsize=8)
        ax.text(i + 0.12, row.r_cov, f"{row.r_cov:.3f}", ha="left", va="bottom", fontsize=8)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(years)
    ax.set_title(f"{mask_kind.title()} flood px vs FEWS ha")
    ax.set_xlabel("Harvest year")
axes[0].set_ylabel("Spearman r (county-level)")
axes[0].text(0.02, 0.05, "left dot = all counties\nright dot = coverage ≥ 0.95", transform=axes[0].transAxes, fontsize=8)

save_fig(
    fig,
    "fig02_spearman_coverage_robust.png",
    "Negative unusual-flood vs harvest rank correlation is stable after dropping low-coverage counties; live explanation is between-county structure (H2), not tile gaps.",
)


## Figure 3 — Panel shrinkage

**Claim:** Stage 7 used 220 county-years while ~1,005 FEWS harvest rows and 914 well-covered county-years are available.

**Falsifier:** If the full panel were already in use, this figure would be flat — it is not.


In [ ]:
# Figure 3
cov = pd.read_csv(OUT / "flood_tile_coverage_admin2.csv")
ha = load_fews_ha_panel()
cov95 = set(cov.loc[cov.flood_tile_coverage_share >= 0.95, "adm2_pcode"])
n_theoretical = cov.adm2_pcode.nunique() * 26
n_fews = len(ha)
n_fews_cov = int(ha.adm2_pcode.isin(cov95).sum())
m, _ = flood_fews_merged()
n_stage7 = len(m)

steps = [
    ("Theoretical county-years\n(79 counties × 26 mask years)", n_theoretical),
    ("FEWS Area Harvested\ncounty-years (pcode matched)", n_fews),
    ("…in counties with tile coverage ≥ 0.95", n_fews_cov),
    ("Stage 7 flood × FEWS panel\n(years hardcoded 2022–2024)", n_stage7),
]
labels, vals = zip(*steps)
y = np.arange(len(steps))
fig, ax = plt.subplots(figsize=(9, 4.2))
colors = ["#bdc3c7", "#85c1e9", "#5dade2", "#e74c3c"]
bars = ax.barh(y, vals, color=colors, edgecolor="white", height=0.55)
for bar, v in zip(bars, vals):
    ax.text(
        bar.get_width() + max(vals) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(v):,}",
        va="center",
        fontsize=10,
        fontweight="bold",
    )
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("County-year count")
ax.set_title("How much panel Stage 7 left on the table")
ratio = n_fews / n_stage7
ax.text(0.98, 0.02, f"FEWS panel ≈ {ratio:.1f}× Stage 7 n", transform=ax.transAxes, ha="right", fontsize=9)

save_fig(
    fig,
    "fig03_panel_shrinkage_waterfall.png",
    "Stage 7 stress-tests used 220 county-years while ~1,005 FEWS harvest rows exist — extend years before claiming within-county flood loss.",
)


## Figure 4 — Detected flood season vs rains

**Claim:** National flood *detections* peak in Dec–Feb, not during Jun–Oct rains — either Sudd drainage lag or wet-season cloud masking.

**Falsifier:** If detections tracked rainfall monthly, H3 seasonal windows would be trivial — they do not.


In [ ]:
# Figure 4
daily = pd.read_csv(
    PROCESSED_DIR / "flood_masks" / "flood_masks_daily_counts_processed.csv",
    parse_dates=["date"],
)
daily["month"] = daily.date.dt.month

def ym_pivot(label: str) -> pd.DataFrame:
    sub = daily[daily.flood_type_label == label]
    return sub.pivot_table(index="year", columns="month", values="flooded_pixel_count", aggfunc="sum")

u = ym_pivot("unusual")
r = ym_pivot("recurring")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, mat, title in zip(axes, [u, r], ["Unusual", "Recurring"]):
    data = mat.fillna(0).to_numpy()
    im = ax.imshow(data, aspect="auto", norm=LogNorm(vmin=1.0, vmax=data.max()), cmap="magma")
    ax.set_title(f"{title} flooded pixel-days (log scale)")
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index, fontsize=7)
    ax.set_xticks(range(12))
    ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    for band, color in [((3, 5), "#27ae60"), ((5, 10), "#3498db"), ((8, 10), "#f39c12")]:
        ax.axvspan(band[0] - 0.5, band[1] + 0.5, color=color, alpha=0.12)
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label="pixel-days / month")
axes[1].set_xlabel("Month")
share_u = u.div(u.sum(axis=1), axis=0).mean() * 100
note = (
    f"Mean unusual monthly share: Dec {share_u.get(12, 0):.1f}%, Jan {share_u.get(1, 0):.1f}%; "
    f"Apr {share_u.get(4, 0):.1f}%. Post-2021 regime jump visible."
)
fig.text(0.5, 0.01, note, ha="center", fontsize=9)

save_fig(
    fig,
    "fig04_flood_detection_seasonality.png",
    "Flood detections peak in dry season (Dec–Feb), inverted vs rains — interpret H3 windows with drainage lag and/or cloud-detection bias; post-2020 magnitude step may contaminate baselines.",
)


## Figure 5 — Why a combined end-to-end model is no-go

**Claim:** Outcome and exposure channels are jointly sparse at county grain; OCHA NaN is unassessed, not zero.

**Falsifier:** If most counties had full OCHA + FEWS + conflict + flood in one row, a unified model would be data-limited only by n — it is not.


In [ ]:
# Figure 5
admin = pd.read_csv(PROCESSED_DIR / "Administrative boundaries" / "ssd_admin2_processed.csv")
admin = admin.rename(columns={"pcode": "adm2_pcode", "name": "adm2_name", "parent_name": "adm1_name"})
ocha = pd.read_csv(OUT / "stage6_ocha_20251130_county.csv").rename(columns={"Admin2_PCODE": "adm2_pcode"})
dtm = pd.read_csv(OUT / "stage6_dtm_r16_county_stock.csv")
cov = pd.read_csv(OUT / "flood_tile_coverage_admin2.csv")
xw = pd.read_csv(OUT / "crosswalks" / "fews_fnid_to_admin2.csv")
fews_pcodes = set(xw.cod_pcode.dropna().unique())
ipc = pd.read_csv(PROCESSED_DIR / "IPC" / "ipc_phase3plus_county_long_processed.csv")
names = pd.read_csv(OUT / "crosswalks" / "admin2_names.csv")
ipc_match = names[names.source == "IPC"][["source_name", "cod_pcode"]].drop_duplicates()
ipc_pcodes = set(
    ipc.merge(ipc_match, left_on="county", right_on="source_name", how="inner")["cod_pcode"].dropna()
)

ged_path = EXT_DATA / "UCDP_georeferenced_and_nonstate" / "ged261-csv" / "GEDEvent_v26_1.csv"
ged = pd.read_csv(ged_path, usecols=["country", "latitude", "longitude"])
ged = ged[ged.country.astype(str).str.contains("South Sudan", case=False, na=False)]
gdf_a = gpd.read_file(ADMIN2_GEO).to_crs("EPSG:4326")
gpts = gpd.GeoDataFrame(
    ged.dropna(subset=["latitude", "longitude"]),
    geometry=gpd.points_from_xy(ged.longitude, ged.latitude),
    crs="EPSG:4326",
)
ucdp_pcodes = set(
    gpd.sjoin(gpts, gdf_a[["adm2_pcode", "geometry"]], how="inner", predicate="within").adm2_pcode.unique()
)

base = admin[["adm2_pcode", "adm2_name", "adm1_name"]].copy()
ocha_aff = ocha.set_index("adm2_pcode")["people_affected"]
ocha_disp = ocha.set_index("adm2_pcode")["people_displaced"]
dtm_pcodes = set(dtm.County_INT_PCode.dropna().unique())

cols = ["ocha_aff", "ocha_disp", "dtm_r16", "fews", "asap", "ipc", "ucdp_ged", "flood_obs"]
mat = pd.DataFrame(index=base.adm2_pcode)
mat["ocha_aff"] = (base.adm2_pcode.map(ocha_aff).notna()).astype(int) * 2
mat["ocha_disp"] = (base.adm2_pcode.map(ocha_disp).notna()).astype(int) * 2
mat["dtm_r16"] = base.adm2_pcode.isin(dtm_pcodes).map({True: 2, False: 0})
mat["fews"] = base.adm2_pcode.isin(fews_pcodes).map({True: 2, False: 0})
mat["asap"] = 2
mat["ipc"] = base.adm2_pcode.isin(ipc_pcodes).map({True: 2, False: 0})
mat["ucdp_ged"] = base.adm2_pcode.isin(ucdp_pcodes).map({True: 2, False: 0})
flood_map = cov.set_index("adm2_pcode")["flood_observed"].to_dict()
mat["flood_obs"] = [2 if flood_map.get(p, False) else 0 for p in base.adm2_pcode]
mat = mat.join(base.set_index("adm2_pcode")[["adm1_name", "adm2_name"]])
mat = mat.sort_values(["adm1_name", "adm2_name"])

fig, ax = plt.subplots(figsize=(8, 14))
data = mat[cols].to_numpy()
im = ax.imshow(data, aspect="auto", cmap="RdYlGn", vmin=0, vmax=2)
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(
    ["OCHA\naffected", "OCHA\ndisplaced", "DTM\nR16", "FEWS", "ASAP", "IPC", "UCDP\nGED", "Flood\nobs"],
    fontsize=9,
)
ax.set_yticks(range(len(mat)))
ax.set_yticklabels(mat.adm2_name, fontsize=6)
counts = {c: int((mat[c] == 2).sum()) for c in cols}
ax.set_title("County × data channel (green=present, red=absent)")
for j, c in enumerate(cols):
    ax.text(j, -1.8, f"{counts[c]}/79", ha="center", fontsize=8, fontweight="bold")
fig.colorbar(im, ax=ax, ticks=[0, 2], label="0 absent · 2 present")

save_fig(
    fig,
    "fig05_county_data_availability_matrix.png",
    "Joint county coverage is sparse (OCHA 38/79 affected, 31/79 displaced) — supports Stage 8 combined-model no-go and H6 blind-spot targeting.",
)


## Figure 6 — Redundancy structure (Stage 4)

**Claim:** Several channels are derived or overlapping; a single model that stacks them double-counts information.

**Falsifier:** If redundancy pairs were independent, the graph would be mostly dotted “independent” edges — it is not.


In [ ]:
# Figure 6
red = pd.read_csv(OUT / "redundancy_matrix.csv")
nodes = sorted(set(red.series_a) | set(red.series_b))
n = len(nodes)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
pos = {node: (np.cos(a), np.sin(a)) for node, a in zip(nodes, angles)}

style_map = {
    "derived": ("-", 1.8, "#8e44ad"),
    "overlapping": ("--", 1.2, "#7f8c8d"),
    "related_same_source": (":", 1.4, "#d35400"),
    "independent": ("-.", 1.0, "#27ae60"),
}

fig, ax = plt.subplots(figsize=(11, 11))
ax.set_aspect("equal")
ax.axis("off")

for _, row in red.iterrows():
    x1, y1 = pos[row.series_a]
    x2, y2 = pos[row.series_b]
    ls, lw, col = style_map.get(row.relation, ("-", 1.0, "#333"))
    ax.plot([x1, x2], [y1, y2], ls=ls, lw=lw, color=col, alpha=0.75, zorder=1)
    if row.series_a == "UCDP_NonState_year_dyad" and row.series_b == "UCDP_GED_type2_events_year":
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my, "Spearman ~0.77\nby year", fontsize=8, ha="center", color="#8e44ad", fontweight="bold")

for node, (x, y) in pos.items():
    ax.scatter([x], [y], s=120, c="#2c3e50", zorder=2)
    label = node.replace("_", " ")
    if len(label) > 22:
        label = label[:20] + "…"
    ax.text(x * 1.12, y * 1.12, label, fontsize=6.5, ha="center", va="center")

legend_elems = [
    mpatches.Patch(color="#8e44ad", label="derived (solid)"),
    mpatches.Patch(color="#7f8c8d", label="overlapping (dashed)"),
    mpatches.Patch(color="#d35400", label="same source (dotted)"),
    mpatches.Patch(color="#27ae60", label="independent (dash-dot)"),
]
ax.legend(handles=legend_elems, loc="lower left", fontsize=8)

save_fig(
    fig,
    "fig06_redundancy_network.png",
    "Stage 4 redundancy: do not stack ACLED+UCDP fatalities or GED type-2 + Non-State; one series per information channel.",
)


## Summary

| Figure | Takeaway |
|--------|----------|
| 1 | ASAP “no flood” group mean = 2 unobservable breadbasket counties |
| 2 | Flood–harvest Spearman survives coverage filter → test H2 panel extension |
| 3 | Stage 7 used 220 of ~1,005 FEWS county-years |
| 4 | Detections peak dry season; post-2020 step warns baseline contamination |
| 5 | Sparse OCHA + rich flood/FEWS → no unified county model |
| 6 | Redundancy rules block naive multi-source stacking |

Next: implement H2/H3 in code (`impact_stages` or a dedicated panel module) per `STAGE9_HYPOTHESES.md`.
